In [1]:
import os
import yaml
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from pathlib import Path
from tqdm import tqdm

print("\n" + "═"*60)
print("  REAL YOLO + GCN INFERENCE & EVALUATION")
print("═"*60)


════════════════════════════════════════════════════════════
  REAL YOLO + GCN INFERENCE & EVALUATION
════════════════════════════════════════════════════════════


In [2]:
WORKING_DIR = "G:/FRP 2026/AlphaDent/working"
RESULT_DIR = os.path.join(WORKING_DIR, "train_run_large_6407")
YOLO_WEIGHTS = os.path.join(RESULT_DIR, "weights/best.pt")
DATA_YAML = "G:/FRP 2026/AlphaDent/yaml/yolo_seg_train.yaml"
IMAGE_DIR = "G:/FRP 2026/AlphaDent/images/train"
LABEL_DIR = "G:/FRP 2026/AlphaDent/labels/train"
VALID_DIR = "G:/FRP 2026/AlphaDent/images/valid"
OUTPUT_DIR = "G:/FRP 2026/AlphaDent/outputs"

# Derived Validation Labels Path
LABEL_DIR_VALID = "G:/FRP 2026/AlphaDent/labels/valid"

COOCCURRENCE_MATRIX_PATH = os.path.join(OUTPUT_DIR, "matrices", "A_norm.npy")

IMAGE_FOLDER = Path(VALID_DIR)
OUTPUT_FOLDER = os.path.join(OUTPUT_DIR, "refined_labels")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [3]:
print(f"Loading YOLO model from {YOLO_WEIGHTS}...")
model = YOLO(YOLO_WEIGHTS)

Loading YOLO model from G:/FRP 2026/AlphaDent/working\train_run_large_6407\weights/best.pt...


In [4]:
class GCN(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features):
        super(GCN, self).__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_features)

    def forward(self, x, A):
        x = torch.matmul(A, x)
        x = F.relu(self.fc1(x))
        x = torch.matmul(A, x)
        x = torch.sigmoid(self.fc2(x))
        return x

print("Loading Co-occurrence Matrix...")
if os.path.exists(COOCCURRENCE_MATRIX_PATH):
    A_static = np.load(COOCCURRENCE_MATRIX_PATH)
else:
    # Fallback to variable from memory if the file doesn't exist yet
    A_static = A_norm.copy() 

A_static_t = torch.tensor(A_static, dtype=torch.float32)

# Normalize adjacency
D = torch.diag(torch.sum(A_static_t, dim=1))
D_inv_sqrt = torch.inverse(torch.sqrt(D + 1e-6))
A_norm_t = D_inv_sqrt @ A_static_t @ D_inv_sqrt

# FIXED: Initialize with 1 feature per node to prevent size mismatch
num_classes = A_static_t.shape[0]
gcn_model = GCN(1, 64, 1)  
gcn_model.eval()

Loading Co-occurrence Matrix...


GCN(
  (fc1): Linear(in_features=1, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)

In [5]:
def extract_predictions(results):
    all_preds = []
    for r in results:
        boxes = r.boxes
        preds = []
        if boxes is not None:
            for b in boxes:
                cls = int(b.cls)
                conf = float(b.conf)
                # IMPORTANT: Use xywhn for normalized coordinates 
                xywhn = b.xywhn[0].tolist() 
                preds.append((cls, conf, xywhn))
        all_preds.append(preds)
    return all_preds

def build_score_vector(preds, num_classes):
    scores = np.zeros(num_classes)
    for cls, conf, _ in preds:
        scores[cls] = max(scores[cls], conf)
    return torch.tensor(scores, dtype=torch.float32)

def build_dynamic_adj(scores, A_static_t):
    scores = scores.unsqueeze(1)
    dynamic_A = A_static_t * (scores @ scores.T)
    D = torch.diag(torch.sum(dynamic_A, dim=1) + 1e-6)
    D_inv_sqrt = torch.inverse(torch.sqrt(D))
    A_norm_dyn = D_inv_sqrt @ dynamic_A @ D_inv_sqrt
    return A_norm_dyn

def refine_predictions(preds, refined_scores):
    refined_preds = []
    for cls, conf, box in preds:
        new_conf = conf * float(refined_scores[cls])
        refined_preds.append((cls, new_conf, box))
    return refined_preds

def save_predictions(preds, save_path):
    with open(save_path, 'w') as f:
        for cls, conf, (cx, cy, w, h) in preds:
            f.write(f"{cls} {cx} {cy} {w} {h} {conf:.5f}\n")

def cxcywh_to_xyxy(boxes):
    """Convert normalized [cx, cy, w, h] to normalized [x1, y1, x2, y2]"""
    if len(boxes) == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    x_c, y_c, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x1, y1 = x_c - w / 2, y_c - h / 2
    x2, y2 = x_c + w / 2, y_c + h / 2
    return torch.stack([x1, y1, x2, y2], dim=1)

def load_gt_boxes(txt_path):
    """Safely extracts Ground Truth bounding boxes from YOLO format (handles both boxes and polygons)"""
    boxes, labels = [], []
    if os.path.exists(txt_path) and os.path.getsize(txt_path) > 0:
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    if len(parts) == 5:  # Bounding Box: cls cx cy w h
                        cx, cy, w, h = map(float, parts[1:])
                        x1, y1, x2, y2 = cx - w/2, cy - h/2, cx + w/2, cy + h/2
                    else:  # Polygon (Segmentation): cls x1 y1 x2 y2 ...
                        coords = np.array(parts[1:], dtype=float).reshape(-1, 2)
                        x1, y1 = coords[:, 0].min(), coords[:, 1].min()
                        x2, y2 = coords[:, 0].max(), coords[:, 1].max()
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls_id)
    if len(boxes) == 0:
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

In [6]:
print("\n--- Running Inference & Building COCO Metrics ---")

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    
    # Initialize COCO evaluators 
    # (We use 'xyxy' because our load_gt_boxes helper converts to xyxy automatically)
    metric_baseline = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    metric_refined = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    
    # 1. Run YOLO Inference
    print("Running YOLO inference on validation set...")
    results = model.predict(source=str(IMAGE_FOLDER), conf=0.001, save=False, verbose=False)
    all_preds = extract_predictions(results)
    
    # 2. Loop through all images for Refinement and Evaluation
    for r, preds_baseline in tqdm(zip(results, all_preds), total=len(results), desc="Refining & Evaluating"):
        img_name = Path(r.path).stem
        gt_path = os.path.join(LABEL_DIR_VALID, f"{img_name}.txt")
        
        # --- A. GCN REFINEMENT ---
        score_vec = build_score_vector(preds_baseline, num_classes)
        A_use = build_dynamic_adj(score_vec, A_static_t)
        
        with torch.no_grad():
            score_vec_input = score_vec.unsqueeze(1) # Handle shape (4,) -> (4, 1)
            refined_scores = gcn_model(score_vec_input, A_use).squeeze()
            
        preds_refined = refine_predictions(preds_baseline, refined_scores)
        
        # Save refined predictions to disk
        save_path = os.path.join(OUTPUT_FOLDER, f"{img_name}.txt")
        save_predictions(preds_refined, save_path)
        
        # --- B. FORMAT FOR TORCHMETRICS ---
        # 1. Ground Truth
        gt_boxes, gt_labels = load_gt_boxes(gt_path)
        target = [dict(
            boxes=gt_boxes,
            labels=gt_labels
        )]
        
        # 2. Baseline Preds
        if len(preds_baseline) > 0:
            # Extract boxes and convert cxcywh to xyxy
            base_boxes = torch.tensor([p[2] for p in preds_baseline], dtype=torch.float32)
            base_boxes = cxcywh_to_xyxy(base_boxes)
            base_scores = torch.tensor([p[1] for p in preds_baseline], dtype=torch.float32)
            base_labels = torch.tensor([p[0] for p in preds_baseline], dtype=torch.int64)
        else:
            base_boxes = torch.empty((0, 4), dtype=torch.float32)
            base_scores = torch.empty((0,), dtype=torch.float32)
            base_labels = torch.empty((0,), dtype=torch.int64)
            
        pred_base = [dict(boxes=base_boxes, scores=base_scores, labels=base_labels)]
        
        # 3. Refined Preds
        if len(preds_refined) > 0:
            # Extract boxes and convert cxcywh to xyxy
            ref_boxes = torch.tensor([p[2] for p in preds_refined], dtype=torch.float32)
            ref_boxes = cxcywh_to_xyxy(ref_boxes)
            ref_scores = torch.tensor([p[1] for p in preds_refined], dtype=torch.float32)
            ref_labels = torch.tensor([p[0] for p in preds_refined], dtype=torch.int64)
        else:
            ref_boxes = torch.empty((0, 4), dtype=torch.float32)
            ref_scores = torch.empty((0,), dtype=torch.float32)
            ref_labels = torch.empty((0,), dtype=torch.int64)
            
        pred_ref = [dict(boxes=ref_boxes, scores=ref_scores, labels=ref_labels)]
        
        # 4. Update metrics
        metric_baseline.update(pred_base, target)
        metric_refined.update(pred_ref, target)
        
    # 3. Compute Final Scores
    print("\nCalculating final COCO metrics (this might take a moment)...")
    base_results = metric_baseline.compute()
    ref_results = metric_refined.compute()
    
    baseline_map50 = base_results['map_50'].item()
    baseline_map = base_results['map'].item()
    
    refined_map50 = ref_results['map_50'].item()
    refined_map = ref_results['map'].item()

except ImportError:
    print("\n[ERROR] 'torchmetrics' is not installed.")
    print("Please run: !pip install torchmetrics pycocotools")
    baseline_map50, baseline_map, refined_map50, refined_map = 0, 0, 0, 0
except Exception as e:
    import traceback
    print(f"\n[ERROR] Evaluation failed: {e}")
    traceback.print_exc()
    baseline_map50, baseline_map, refined_map50, refined_map = 0, 0, 0, 0


--- Running Inference & Building COCO Metrics ---
Running YOLO inference on validation set...


Refining & Evaluating:   0%|          | 0/83 [00:00<?, ?it/s]e:\Anaconda\envs\yolo\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
Refining & Evaluating: 100%|██████████| 83/83 [00:00<00:00, 138.39it/s]



Calculating final COCO metrics (this might take a moment)...


In [7]:
print("\n" + "═"*60)
print("  FINAL EVALUATION REPORT: Baseline vs GCN-Improved")
print("═"*60)
print(f"  Metric       | Baseline YOLO  | YOLO + GCN     | Improvement")
print("  " + "-"*56)
diff_map50 = refined_map50 - baseline_map50
diff_map = refined_map - baseline_map
marker50 = "▲" if diff_map50 > 0 else "▼" if diff_map50 < 0 else "="
marker = "▲" if diff_map > 0 else "▼" if diff_map < 0 else "="

print(f"  mAP@50       | {baseline_map50:.4f}         | {refined_map50:.4f}         | {marker50} {abs(diff_map50):.4f}")
print(f"  mAP@50-95    | {baseline_map:.4f}         | {refined_map:.4f}         | {marker} {abs(diff_map):.4f}")
print("═"*60)
print(f"Pipeline completed: Refined labels saved to {OUTPUT_FOLDER}")


════════════════════════════════════════════════════════════
  FINAL EVALUATION REPORT: Baseline vs GCN-Improved
════════════════════════════════════════════════════════════
  Metric       | Baseline YOLO  | YOLO + GCN     | Improvement
  --------------------------------------------------------
  mAP@50       | 0.6163         | 0.6171         | ▲ 0.0008
  mAP@50-95    | 0.4343         | 0.4347         | ▲ 0.0005
════════════════════════════════════════════════════════════
Pipeline completed: Refined labels saved to G:/FRP 2026/AlphaDent/outputs\refined_labels
